# Azure Operational LLM Serving: From Adapter to Evidence

> **The story:** In 2007, Michael Nygard named the circuit-breaker pattern in *Release It!* to stop one failing dependency from consuming an entire system. In 2016, Betsy Beyer, Chris Jones, Jennifer Petoff, and Niall Richard Murphy made service-level objectives and error budgets operational practice in Google's *Site Reliability Engineering*. Model serving inherits both lessons: a useful model is not a service until failure is bounded and evidence controls release.
>
> **Where you are:** Riverside has a LoRA editing candidate, gateway mechanisms, a quantization decision process, and inference metrics. It does not yet have one stable endpoint that proves readiness, rejects overload, deduplicates work, respects deadlines, rolls back immutably, and reports tail latency.
>
> **Notation:** $C$ is admitted concurrency; $Q$ is queue time; $D$ is an absolute request deadline; $T_{first}$ is TTFT; $T_{token}$ is TPOT; $p_{95}(X)$ is the 95th percentile of measurement $X$.

> **Evidence banner:** `LOCAL` mechanisms, `SUBSTITUTED` model work, `UNVALIDATED` Azure behavior.

The notebook starts only an ephemeral loopback HTTP server. `SyntheticBackend` sleeps on the real clock, fails on configured calls, and returns contract-shaped metadata. It does not load a transformer or claim text quality. The Azure ML/APIM adapter builds a request plan but always blocks network traffic.

> **Inter-notebook contract:** The [fine-tuning owner](../../genai/03-llm-finetuning/03-llm-finetuning-comparison-and-decision.ipynb) supplies the candidate and release-evidence discipline. The [gateway owner](../../genai/06-llm-gateway/06-llm-gateway.ipynb), [quantization owner](../06-quantization/quantization-in-depth.ipynb), and [inference owner](../07-inference-systems/inference-systems.ipynb) supply the mechanisms composed here. This chapter reloads fixtures from disk and delivers local operational measurements plus a production adapter boundary.

## Table of Contents

1. [The Challenge](#0--the-challenge)
2. [Contract Before Service](#1--contract-before-service)
3. [Bound Concurrency](#2--bound-concurrency)
4. [Deduplicate and Account](#3--deduplicate-and-account)
5. [Deadlines, Retries, Circuit Breaking, and Fallback](#4--deadlines-retries-circuit-breaking-and-fallback)
6. [Immutable Releases and Blue/Green](#5--immutable-releases-and-bluegreen)
7. [Tail Latency, TTFT, TPOT, Traces, and SLO Gates](#6--tail-latency-ttft-tpot-traces-and-slo-gates)
8. [One Config, Two Endpoint Adapters](#7--one-config-two-endpoint-adapters)
9. [Coverage and Decision](#8--coverage-and-decision)

> If links do not scroll in your viewer, use the notebook outline or `Ctrl+F` with the section title.

## 0 · The Challenge

> **The mission**: Riverside House - serve `riverside-editor` behind one stable contract, keep local p95 total latency at or below 200 ms, and reject overload before it becomes a timeout storm.

**What we know so far:**
- Fine-tuning produced one LoRA candidate, but artifact presence is not a production decision.
- The gateway owner built normalization, limiting, fallback, and caching mechanisms.
- The inference owner separated queue time, TTFT, TPOT, and throughput.
- Current operational proof is **0 of 7 required serving failure classes**; no local p95 sample exists yet.
- **But we still cannot prove that a process serving the adapter is ready, bounded, recoverable, or releasable.**

**What's blocking us:** An in-process call has no wire contract. A naive HTTP process can report alive before its artifact is valid, admit work it cannot finish, repeat generation, spend beyond the caller's deadline, and mutate a release with no rollback target. Average latency can remain reassuring while a small tail fails users.

**What this chapter unlocks:** Authored checks for all 7 local failure classes, a measured p95 comparison against the 200 ms local threshold when run, and an exact list of Azure claims that remain `UNVALIDATED`.

```mermaid
flowchart LR
    A["Saved adapter\nin-process"] --> B["Failure: no stable\nservice contract"]
    B --> C["Local HTTP +\nfrozen v1 contracts"]
    C --> D["Failure: concurrency, duplicates,\ndeadlines, release drift"]
    D --> E["Bounded service +\nrelease evidence"]
    style A fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style B fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style C fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style D fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style E fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```

### Complaint chain

| Attempt | What it proves | Complaint that forces the next step |
|---|---|---|
| 1. Call the adapter in-process | One function can return | No stable request, error, readiness, or release contract |
| 2. Put HTTP in front | A process can accept a request | It accepts cold traffic and hides queue overload |
| 3. Bound readiness and admission | Early and excess traffic can be rejected | Duplicate work, missing usage, and unbounded recovery remain |
| 4. Add idempotency and resilience | Work and failure are bounded | A changed release and tail latency can still pass by average |
| Final form | Immutable routing plus percentile gates | Local evidence still cannot prove Azure behavior |

| Sub-topic | Coverage | Reason |
|---|---|---|
| Contracts, readiness, admission, overload | Built; proof cells authored | First service-boundary failures |
| Idempotency, accounting, deadline, retry, circuit, fallback | Built; proof cells authored | Failures compound under traffic |
| Immutable manifest, blue/green routing, tail-latency gates | Built; proof cells authored | Release behavior must be reversible and measured |
| OTel export, Redis, streaming SSE | Explained and illustrated | External infrastructure is not started here |
| Entra ID, APIM execution, Azure ML scaling | Named only | Requires authorized Azure evidence |

| Teaching scale | Local lab | Azure production target |
|---|---|---|
| Deterministic synthetic backend | Loopback contract-shaped endpoint | Azure ML managed online endpoint |
| In-process policy objects | Measured HTTP behavior when run | APIM plus orchestrator policy |
| Local metric records | Bounded attribute validation when run | Application Insights/Azure Monitor export |

In [ ]:
# ── Load Local Fixtures and the Frozen Contract Registry ─────────────────
from copy import deepcopy
from dataclasses import replace
import json
from pathlib import Path
from statistics import mean
import random
import sys
import time


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (
            (candidate / "AUTHORING_GUIDE.md").is_file()
            and (candidate / "learning").is_dir()
            and (candidate / "projects").is_dir()
        ):
            return candidate
    raise FileNotFoundError("Run inside the ai-portfolio repository.")


REPO_ROOT = find_repo_root(Path.cwd().resolve())
CHAPTER_DIR = REPO_ROOT / "learning/ai-infrastructure/09-azure-operational-llm-serving"
FIXTURE_DIR = CHAPTER_DIR / "fixtures"
CONTRACT_DIR = REPO_ROOT / "projects/riverside-ai-platform/contracts/v1"
sys.path.insert(0, str(CHAPTER_DIR / "scripts"))

from serving_lab import (
    ArtifactValidationError, CloudCallBlocked, RiversideService, ServicePolicy,
    SyntheticBackend, build_endpoint_adapter, load_json, run_concurrent_requests,
    start_local_server, summarize_metrics, validate_contract,
    validate_release_invariants,
)

local_config = load_json(FIXTURE_DIR / "deployment.local.json")
release_manifest = load_json(FIXTURE_DIR / "model-release-manifest.json")
safe_request = load_json(FIXTURE_DIR / "privacy-safe-request.json")
slo_policy = load_json(FIXTURE_DIR / "operational-slo-policy.json")
local_policy = ServicePolicy.from_config(local_config)
release_schema = CONTRACT_DIR / "model-release-manifest.schema.json"
request_schema = CONTRACT_DIR / "app-chat-completion-request.schema.json"
response_schema = CONTRACT_DIR / "app-chat-completion-response.schema.json"
error_schema = CONTRACT_DIR / "app-error.schema.json"
telemetry_schema = CONTRACT_DIR / "telemetry-attributes.schema.json"

print("Evidence: LOCAL mechanisms | SUBSTITUTED model | UNVALIDATED Azure")
print(f"Loaded release {release_manifest['release_id']} for alias {safe_request['model']}.")
print("No Azure SDK, credential, transformer, Docker service, or external endpoint is used.")

### Code Walkthrough: Load the Serving Boundary

1. **Repository discovery** - `find_repo_root()` requires the durable repository markers `AUTHORING_GUIDE.md`, `learning/`, and `projects/`, so opening the notebook from the repository root or chapter directory resolves the same fixtures.
2. **Frozen contracts** - request, response, error, release, and telemetry schemas come from the shared v1 registry. The notebook consumes that boundary; it does not redefine it.
3. **One running release** - every experiment reuses `riverside-editor-tutorial-2026-08-05`, the same public request, and the same `riverside-editor` alias. Only the failure variable changes.
4. **Two adapters** - `build_endpoint_adapter()` selects a loopback HTTP adapter or a network-blocked Azure ML/APIM request planner from configuration. The application-facing request stays unchanged.
5. **Evidence labels** - `LOCAL`, `SUBSTITUTED`, and `UNVALIDATED` are conclusions about scope, not decoration. A later run may promote a local mechanism to local measured evidence; it cannot promote Azure behavior.

> **Shape note:** The local response and Azure request plan are JSON objects at the same application boundary. They are not equivalent runtime implementations; only their contract-facing inputs are intentionally parallel.

**Checkpoint:** The chapter is pinned to the local v1 schema registry and privacy-safe fixtures. No operational number has been claimed; every number below is produced by the local clock when the notebook is eventually run.

## 1 · Contract Before Service

Riverside's adapter can return a string in-process. That proves only that one function returned. It does not define request limits, normalized errors, usage, citations, deployment identity, or when traffic is safe.

```mermaid
flowchart LR
    A["Python call works"] --> B["No wire contract"]
    B --> C["Validate release + request"]
    C --> D{"Warm-up complete?"}
    D -->|no| X["503 release_unavailable"]
    D -->|yes| E["Ready for traffic"]
    style A fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style B fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style C fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style D fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style X fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style E fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```

**Predict:** Which is strongest evidence: one in-process completion, `/health` returning 200 while cold, or `/ready` changing from 503 to 200 only after contract validation and warm-up?

| Local mechanism | Azure boundary | Revalidate in cloud |
|---|---|---|
| Schema registry and readiness gate | Artifact registration and Azure ML probes | Artifact access, digest verification, image/model warm-up, probe semantics |

**Common Pitfalls**

| | Pattern | Why it matters |
|---|---|---|
| Wrong | Mark ready when a port binds | Traffic reaches a cold or invalid release |
| Right | Validate immutable unit, warm, then mark ready | Admission starts only when usable |
| Wrong | Use `latest` | Rollback cannot reconstruct behavior |
| Right | Pin revision, digest, runtime, source commit | Evidence identifies exactly what ran |

**Quick Health Check:** validate release/request, reject `latest`, require 503 before warm-up and 200 afterward.

In [ ]:
# ── Measure In-Process Insufficiency and the Readiness Transition ─────────
direct_backend = SyntheticBackend("direct", first_token_ms=8, tpot_ms=0.5)
started = time.perf_counter()
direct_result = direct_backend.generate(safe_request, deadline_at=time.perf_counter() + 1)
direct_ms = (time.perf_counter() - started) * 1000
print(f"In-process completion: {direct_ms:.2f} ms; no HTTP/readiness evidence.")

validate_contract(release_manifest, release_schema, CONTRACT_DIR)
validate_release_invariants(release_manifest)
validate_contract(safe_request, request_schema, CONTRACT_DIR)
mutable_release = deepcopy(release_manifest)
mutable_release["version"] = "latest"
try:
    validate_contract(mutable_release, release_schema, CONTRACT_DIR)
    validate_release_invariants(mutable_release)
    mutable_rejected = False
except ArtifactValidationError:
    mutable_rejected = True

readiness_service = RiversideService(release_manifest, SyntheticBackend("ready"), local_policy)
readiness_server = start_local_server(readiness_service)
readiness_adapter = build_endpoint_adapter(local_config, endpoint_override=readiness_server.base_url)
before_ready = readiness_adapter.invoke(safe_request)
readiness_service.validate_and_warm(release_schema, CONTRACT_DIR)
after_ready = readiness_adapter.invoke(safe_request, idempotency_key="ready-check")
readiness_server.close()

lifecycle_checks = {
    "mutable_rejected": mutable_rejected,
    "cold_is_503": before_ready.status == 503,
    "warm_is_200": after_ready.status == 200,
}
for name, passed in lifecycle_checks.items():
    print(f"{'PASS' if passed else 'FAIL'}: {name}")
if all(lifecycle_checks.values()):
    print("Prediction confirmed: contract validation plus warm-up produces the 503 to 200 transition.")
else:
    print("Prediction contradicted: readiness evidence is incomplete; do not admit traffic.")

**Checkpoint:** The boundary advanced from a function return to frozen v1 request/release validation and a measured readiness transition.

**Reflection:** Readiness prevents early traffic, but it says nothing about how much ready traffic the process can finish.

## 2 · Bound Concurrency

The [inference owner](../07-inference-systems/inference-systems.ipynb) showed why queue time belongs inside user-visible latency. A long queue keeps accepting work it cannot finish; a bounded queue returns a short, retryable overload instead.

```mermaid
flowchart LR
    A["Concurrent requests"] --> B{"Capacity C available?"}
    B -->|yes| C["Backend work"]
    B -->|wait without bound| X["Tail latency + timeout storm"]
    B -->|bounded wait| D["429 overloaded"]
    C --> E["Measured completion"]
    style A fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style B fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style C fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style X fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style D fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style E fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```

**Predict:** With $C=1$ and eight simultaneous requests, does a one-second queue or a 10 ms bounded queue give the caller clearer evidence?

| Local mechanism | Azure boundary | Revalidate in cloud |
|---|---|---|
| Semaphore, queue budget, normalized 429 | Azure ML concurrency plus APIM/orchestrator admission | Replica limits, token capacity, autoscale, quota, retry-after behavior |

**Common Pitfalls**

| | Pattern | Why it matters |
|---|---|---|
| Wrong | Limit HTTP connections only | Streaming work can hold model slots for seconds |
| Right | Bound active work, tokens, and wait | Admission reflects scarce resources |
| Wrong | Retry 429 immediately | Retries amplify overload |
| Right | Return retry guidance and jitter | Callers can shed or delay work |

**Quick Health Check:** require only 200/429, normalized overload, and bounded queue evidence.

In [ ]:
# ── Compare a Long Queue with Bounded Admission ───────────────────────────
queue_payloads = []
for index in range(8):
    payload = deepcopy(safe_request)
    payload["messages"][-1]["content"] += f" Public sample {index}."
    queue_payloads.append(payload)


def run_queue_case(name: str, queue_ms: int):
    policy = replace(
        local_policy, max_concurrency=1, max_queue_ms=queue_ms, request_deadline_ms=2000,
        cache_enabled=False, singleflight_enabled=False,
    )
    service = RiversideService(release_manifest, SyntheticBackend(name), policy)
    service.validate_and_warm(release_schema, CONTRACT_DIR)
    server = start_local_server(service)
    adapter = build_endpoint_adapter(local_config, endpoint_override=server.base_url)
    responses = run_concurrent_requests(adapter, queue_payloads, workers=8)
    server.close()
    return service, responses, summarize_metrics(service.metrics)


long_queue_service, long_queue_responses, long_queue_summary = run_queue_case("long-queue", 1000)
bounded_service, bounded_responses, bounded_summary = run_queue_case("bounded-queue", 10)
long_rejections = sum(response.status == 429 for response in long_queue_responses)
bounded_rejections = sum(response.status == 429 for response in bounded_responses)
print("Long queue:", json.dumps(long_queue_summary, indent=2))
print("Bounded queue:", json.dumps(bounded_summary, indent=2))
if bounded_rejections > long_rejections and bounded_summary["p95_queue_ms"] < long_queue_summary["p95_queue_ms"]:
    print("Prediction confirmed: the 10 ms queue returns explicit overload before the one-second queue hides it in latency.")
else:
    print("Prediction contradicted: this run did not separate admission clearly; inspect scheduler noise and repeat before concluding.")

**Your turn:** Change one queue budget and predict whether success rate, p95 queue time, or both move. This changes admission policy, not capacity.

In [ ]:
# ── Change One Queue Variable and Check Both Failure Directions ───────────
QUEUE_MS = 20  # CHANGE THIS: try 5, 20, or 100
exercise_service, exercise_responses, exercise_summary = run_queue_case("queue-drill", QUEUE_MS)
exercise_overloads = [response.body for response in exercise_responses if response.status == 429]
exercise_successes = [response for response in exercise_responses if response.status == 200]
admission_checks = {
    "known_statuses": all(response.status in {200, 429} for response in exercise_responses),
    "bounded_wait_reduces_p95_queue": exercise_summary["p95_queue_ms"] < long_queue_summary["p95_queue_ms"],
    "too_tight_risk_exposed": bool(exercise_overloads),
    "legitimate_work_still_served": bool(exercise_successes),
    "normalized_code": all(body["error"]["code"] == "overloaded" for body in exercise_overloads),
    "retry_guidance": all(body["error"]["retry_after_seconds"] >= 1 for body in exercise_overloads),
}
print(f"Queue budget: {QUEUE_MS} ms; HTTP 200={len(exercise_successes)}, HTTP 429={len(exercise_overloads)}")
print(f"p95 queue: {exercise_summary['p95_queue_ms']:.2f} ms")
for name, passed in admission_checks.items():
    print(f"{'PASS' if passed else 'FAIL'}: {name}")
print("Prediction closure: lowering the budget rejects sooner; raising it admits more wait without adding capacity.")

**Checkpoint:** Unsustainable work becomes explicit 429 evidence instead of hidden queue delay.

**Reflection:** Capacity is bounded, but the service still pays repeatedly when callers submit identical work.

## 3 · Deduplicate and Account

The [gateway owner](../../genai/06-llm-gateway/06-llm-gateway.ipynb) established caching as a cost mechanism. Serving adds a race: concurrent requests can all miss before the first result arrives. Single-flight lets one request perform the work while the others reuse it.

```mermaid
flowchart LR
    A["Six identical requests"] --> B{"Single-flight key?"}
    B -->|no| X["Six backend calls + zero usage"]
    B -->|yes| C["One backend call"]
    C --> D["Five cache hits"]
    D --> E["Token-accounted response"]
    style A fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style B fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style X fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style C fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style D fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style E fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```

**Predict:** Does a normal response cache collapse six simultaneous misses automatically?

| Local mechanism | Azure boundary | Revalidate in cloud |
|---|---|---|
| Release-aware key, in-memory single-flight, usage invariant | Shared application/Redis cache and real tokenizer accounting | Cross-replica consistency, TTL, eviction, encryption, tokenizer parity |

**Common Pitfalls**

| | Pattern | Why it matters |
|---|---|---|
| Wrong | Cache by raw prompt only | Release/index changes can serve stale answers |
| Right | Key by request plus immutable context | Validity follows deployable state |
| Wrong | Put IDs or content in metric labels | Privacy and cardinality fail |
| Right | Keep trace IDs in context; labels on allowlist | Correlation remains bounded |

**Quick Health Check:** require one backend call, five hits, non-zero usage, valid response, and valid telemetry labels.

In [ ]:
# ── Compare Duplicate Work Before and After Single-Flight ─────────────────
duplicate_payloads = [deepcopy(safe_request) for _ in range(6)]


def run_duplicate_case(name: str, cache: bool, singleflight: bool, account: bool):
    backend = SyntheticBackend(name)
    policy = replace(
        local_policy, max_concurrency=6, cache_enabled=cache,
        singleflight_enabled=singleflight, account_tokens=account,
    )
    service = RiversideService(release_manifest, backend, policy)
    service.validate_and_warm(release_schema, CONTRACT_DIR)
    server = start_local_server(service)
    adapter = build_endpoint_adapter(local_config, endpoint_override=server.base_url)
    responses = run_concurrent_requests(
        adapter, duplicate_payloads, workers=6, shared_idempotency_key="same-public-edit"
    )
    server.close()
    return backend, service, responses, summarize_metrics(service.metrics)


broken_backend, broken_service, broken_responses, broken_summary = run_duplicate_case(
    "duplicate-broken", False, False, False
)
fixed_backend, fixed_service, fixed_responses, fixed_summary = run_duplicate_case(
    "duplicate-fixed", True, True, True
)
broken_tokens = sum(response.body["usage"]["total_tokens"] for response in broken_responses)
fixed_tokens = sum(response.body["usage"]["total_tokens"] for response in fixed_responses)
print(f"Backend calls before/after: {broken_backend.call_count} -> {fixed_backend.call_count}")
print(f"Cache hits after: {fixed_summary['cache_hits']}")
print(f"Reported tokens before/after: {broken_tokens} -> {fixed_tokens}")
if broken_backend.call_count > 1 and fixed_backend.call_count == 1:
    print("Prediction confirmed: a response cache alone does not collapse simultaneous misses; single-flight does.")
else:
    print("Prediction contradicted: duplicate work did not separate cleanly; inspect the request key and concurrency path.")

In [ ]:
# ── Run Response and Bounded-Telemetry Health Checks ─────────────────────
validate_contract(fixed_responses[0].body, response_schema, CONTRACT_DIR)
attributes = fixed_service.metrics[-1].metric_attributes(fixed_responses[0].body["deployment"])
validate_contract(attributes, telemetry_schema, CONTRACT_DIR)
dedup_checks = {
    "one_backend_call": fixed_backend.call_count == 1,
    "five_cache_hits": fixed_summary["cache_hits"] == 5,
    "usage_present": fixed_tokens > 0,
    "usage_sums": all(
        response.body["usage"]["total_tokens"]
        == response.body["usage"]["prompt_tokens"] + response.body["usage"]["completion_tokens"]
        for response in fixed_responses
    ),
    "trace_not_metric_label": "trace_id" not in attributes,
}
for name, passed in dedup_checks.items():
    print(f"{'PASS' if passed else 'FAIL'}: {name}")
print("Takeaway: reuse is valid only when backend collapse, usage invariants, response shape, and bounded labels all agree.")

**Checkpoint:** Concurrent repeats move from multiple backend calls and zero usage evidence to one call, release-aware reuse, and contract-valid accounting.

**Reflection:** The remaining backend call can still hang or fail. Without one absolute deadline and bounded recovery, cost is protected while the user is not.

## 4 · Deadlines, Retries, Circuit Breaking, and Fallback

A timeout on each hop is not a request deadline. Queueing, attempts, fallback, and decode all spend from one budget: $D_{remaining}=D_{total}-Q-T_{attempts}-T_{fallback}$. In plain English, every wait leaves less time for useful generation. Retry only idempotent work with budget remaining. A circuit breaker stops attempts on a dependency already known to be failing.

```mermaid
flowchart LR
    A["Request with deadline D"] --> B{"Primary circuit closed?"}
    B -->|yes| C["Bounded attempt"]
    C -->|retryable + budget| C
    C -->|threshold reached| X["Open circuit"]
    B -->|no| D["Reviewed fallback"]
    X --> D
    D --> E["Success or normalized failure"]
    style A fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style B fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style C fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style X fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style D fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style E fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```

**Predict:** After two primary failures at a threshold of two, should the next request call primary again or go directly to the reviewed fallback?

| Local mechanism | Azure boundary | Revalidate in cloud |
|---|---|---|
| Absolute clock deadline, bounded retry, breaker, explicit fallback | APIM policies plus orchestrator/backend resilience | Policy ordering, deadline propagation, idempotency, fallback quality/safety |

**Common Pitfalls**

| | Pattern | Why it matters |
|---|---|---|
| Wrong | Fresh full timeout per retry | One request exceeds the user budget |
| Right | One deadline through every phase | Recovery cannot outlive caller |
| Wrong | Hide fallback as primary | Quality regressions vanish inside success rate |
| Right | Record deployment/fallback evidence | Reliability and quality stay separate |

**Quick Health Check:** normalize timeout, cap retries, open circuit at threshold, stop primary calls, serve reviewed fallback.

In [ ]:
# ── Measure Deadline, Retry, Circuit, and Fallback Paths ──────────────────
deadline_service = RiversideService(
    release_manifest, SyntheticBackend("slow", first_token_ms=80),
    replace(local_policy, request_deadline_ms=30, max_retries=0, cache_enabled=False),
)
deadline_service.validate_and_warm(release_schema, CONTRACT_DIR)
deadline_server = start_local_server(deadline_service)
deadline_adapter = build_endpoint_adapter(local_config, endpoint_override=deadline_server.base_url)
deadline_response = deadline_adapter.invoke(safe_request, idempotency_key="deadline")
deadline_server.close()

primary_backend = SyntheticBackend("primary", fail_first=2)
fallback_backend = SyntheticBackend("reviewed-fallback", first_token_ms=6, tpot_ms=0.5)
resilience_policy = replace(local_policy, max_retries=1, circuit_failure_threshold=2, cache_enabled=False)
resilience_service = RiversideService(
    release_manifest, primary_backend, resilience_policy, fallback_backend=fallback_backend
)
resilience_service.validate_and_warm(release_schema, CONTRACT_DIR)
resilience_server = start_local_server(resilience_service)
resilience_adapter = build_endpoint_adapter(local_config, endpoint_override=resilience_server.base_url)
first_resilient = resilience_adapter.invoke(safe_request, idempotency_key="retry-1")
second_resilient = resilience_adapter.invoke(safe_request, idempotency_key="retry-2")
resilience_server.close()
print(f"Deadline: HTTP {deadline_response.status}, {deadline_response.body['error']['code']}")
print(f"Primary calls: {primary_backend.call_count}; fallback calls: {fallback_backend.call_count}")
print(f"Circuit: {resilience_service.circuit_state}; statuses: {first_resilient.status}, {second_resilient.status}")
if primary_backend.call_count == 2 and resilience_service.circuit_state == "open" and fallback_backend.call_count == 2:
    print("Prediction confirmed: after two failures, the next request bypasses primary and uses the reviewed fallback.")
else:
    print("Prediction contradicted: the breaker or fallback path did not match the stated threshold.")

**Your turn:** Change one absolute deadline around a backend whose first token takes 50 ms. Predict the transition, then let status decide. Scheduler overhead means the practical boundary can be slightly above backend delay.

In [ ]:
# ── Change One Deadline and Run Resilience Health Checks ─────────────────
REQUEST_DEADLINE_MS = 60  # CHANGE THIS: try 40, 60, or 100
drill_service = RiversideService(
    release_manifest, SyntheticBackend("deadline-drill", first_token_ms=50, tpot_ms=1, completion_tokens=6),
    replace(local_policy, request_deadline_ms=REQUEST_DEADLINE_MS, cache_enabled=False),
)
drill_service.validate_and_warm(release_schema, CONTRACT_DIR)
drill_server = start_local_server(drill_service)
drill_adapter = build_endpoint_adapter(local_config, endpoint_override=drill_server.base_url)
drill_response = drill_adapter.invoke(safe_request, idempotency_key="deadline-drill")
drill_server.close()
validate_contract(deadline_response.body, error_schema, CONTRACT_DIR)
resilience_checks = {
    "deadline_normalized": deadline_response.status == 504 and deadline_response.body["error"]["code"] == "timeout",
    "retry_bounded": max(metric.retry_count for metric in resilience_service.metrics) <= 1,
    "circuit_opened": resilience_service.circuit_state == "open",
    "primary_stopped": primary_backend.call_count == 2,
    "fallback_served_both": fallback_backend.call_count == 2 and first_resilient.status == second_resilient.status == 200,
}
print(f"Drill deadline {REQUEST_DEADLINE_MS} ms -> HTTP {drill_response.status}")
if drill_response.status == 200:
    print("Your-turn closure: the deadline left enough budget for first token and decode.")
elif drill_response.status == 504:
    print("Your-turn closure: the deadline expired; increase budget or reduce work before retrying.")
else:
    print("Your-turn closure: an unexpected status needs diagnosis before changing the deadline again.")
for name, passed in resilience_checks.items():
    print(f"{'PASS' if passed else 'FAIL'}: {name}")

**Checkpoint:** Slow work produces a normalized deadline error; primary attempts stop at the breaker threshold; the explicit fallback remains observable.

**Reflection:** Resilience keeps one release available. It does not prove that a changed adapter, tokenizer, runtime, or policy is safe.

## 5 · Immutable Releases and Blue/Green

The fine-tuning owner concludes that artifact presence is not promotion. The [quantization owner](../06-quantization/quantization-in-depth.ipynb) adds that format, runtime, hardware, tokenizer, and precision form one deployable unit. Blue/green preserves that unit while traffic changes separately.

```mermaid
flowchart LR
    A["Immutable manifest"] --> B["Blue accepted"]
    A --> C["Green candidate"]
    D["Stable model alias"] --> E{"Weighted route"}
    E --> B
    E --> C
    C -->|gate fails| X["Green weight 0"]
    style A fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style B fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style C fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style D fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style E fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style X fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```

**Predict:** Which rollback is reconstructable: mutating `latest` by hand, or setting green weight to zero while blue's immutable manifest stays warm?

| Local mechanism | Azure boundary | Revalidate in cloud |
|---|---|---|
| Two manifests, seeded weighted route, zero-green rollback | Azure ML blue/green deployments plus APIM routing | Actual split, drain, rollback time, sticky sessions, retained evidence |

**Common Pitfalls**

| | Pattern | Why it matters |
|---|---|---|
| Wrong | Change files inside a running release | Identity no longer describes behavior |
| Right | New digest/version and slot | Every change has a new identity |
| Wrong | Promote because it loads | Quality, safety, operations, cost absent |
| Right | Require all release-evidence domains | Promotion follows retained evidence |

**Quick Health Check:** validate both manifests, reject `latest`, keep IDs distinct, preserve weights total 100, prove rollback sends zero green.

In [ ]:
# ── Validate Two Immutable Releases and Measure Traffic Allocation ────────
green_manifest = deepcopy(release_manifest)
green_manifest["release_id"] = "riverside-editor-tutorial-green"
green_manifest["version"] = "1.1.0"
green_manifest["adapter"]["digest"]["value"] = "5" * 64
green_manifest["evaluation"]["report_digest"]["value"] = "6" * 64
validate_contract(green_manifest, release_schema, CONTRACT_DIR)
validate_release_invariants(green_manifest)
bad_green = deepcopy(green_manifest)
bad_green["version"] = "latest"
try:
    validate_contract(bad_green, release_schema, CONTRACT_DIR)
    validate_release_invariants(bad_green)
    bad_green_rejected = False
except ArtifactValidationError:
    bad_green_rejected = True


def route_slots(weights: dict[str, int], requests: int, seed: int = 2026):
    rng = random.Random(seed)
    slots = list(weights)
    chosen = rng.choices(slots, weights=[weights[slot] for slot in slots], k=requests)
    return {slot: chosen.count(slot) for slot in slots}


canary_weights = {"blue": 90, "green": 10}
canary_observed = route_slots(canary_weights, 1000)
rollback_observed = route_slots({"blue": 100, "green": 0}, 1000)
print("Canary:", canary_observed)
print("Rollback:", rollback_observed)
if bad_green_rejected and rollback_observed["green"] == 0:
    print("Prediction confirmed: immutable blue plus zero green weight gives a reconstructable rollback.")
else:
    print("Prediction contradicted: release identity or rollback routing is not reconstructable.")

**Your turn:** Change `GREEN_WEIGHT` and predict the range across 1,000 seeded routes. This is a routing drill, not a cloud canary or quality evaluation.

In [ ]:
# ── Change One Canary Weight and Run Release Health Checks ────────────────
GREEN_WEIGHT = 10  # CHANGE THIS: try 1, 10, or 25
my_observed = route_slots({"blue": 100 - GREEN_WEIGHT, "green": GREEN_WEIGHT}, 1000)
expected_green = GREEN_WEIGHT * 10
seeded_tolerance = 40
release_checks = {
    "distinct_ids": release_manifest["release_id"] != green_manifest["release_id"],
    "distinct_digests": release_manifest["adapter"]["digest"] != green_manifest["adapter"]["digest"],
    "latest_rejected": bad_green_rejected,
    "weights_total_100": 0 <= GREEN_WEIGHT <= 100,
    "seeded_route_near_weight": abs(my_observed["green"] - expected_green) <= seeded_tolerance,
    "rollback_zero_green": rollback_observed["green"] == 0,
}
print(f"Configured {GREEN_WEIGHT}% green; observed {my_observed['green']} of 1000.")
print(f"Your-turn closure: expected about {expected_green} green routes within +/-{seeded_tolerance} for this drill.")
for name, passed in release_checks.items():
    print(f"{'PASS' if passed else 'FAIL'}: {name}")

**Checkpoint:** The candidate is a second immutable manifest, not a mutation of blue. Routing and rollback are separate measured configuration changes.

**Reflection:** Reversible routing is necessary but averages can still approve a release while slow users cluster in the tail.

## 6 · Tail Latency, TTFT, TPOT, Traces, and SLO Gates

Averages reward the common path. Riverside's users feel the slow path. The local policy therefore gates the distribution: success rate, rejection rate, p95 total latency, p95 TTFT, and p95 TPOT remain separate.

For ordered measurements $x_{(1)} \le \dots \le x_{(N)}$, this chapter uses a linearly interpolated $p_{95}(X)$. In plain English, p95 estimates the boundary below which roughly 95% of this named sample fell; it is not a production guarantee and ten requests cannot support a p99 claim.

```mermaid
flowchart LR
    A["Ten local requests"] --> B["Mean latency"]
    A --> C["p95 total latency"]
    A --> D["p95 TTFT and TPOT"]
    B --> E{"Mean passes?"}
    C --> F{"Tail passes?"}
    D --> G{"Generation stages pass?"}
    E --> H["Average can look healthy"]
    F -->|no| X["HOLD local release"]
    G -->|no| X
    F -->|yes| J["Retain local evidence"]
    G -->|yes| J
    style A fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style B fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style C fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style D fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style E fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style F fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style G fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style H fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style X fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style J fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```

**Predict:** Ten requests include eight fast first tokens and two 220 ms first tokens. Which result is most likely?

1. Mean and p95 both pass the 200 ms total-latency gate.
2. Mean passes while p95 fails, so the release remains on hold.
3. Mean and p95 both fail.

| Local mechanism | Azure boundary | Revalidate in cloud |
|---|---|---|
| Real loopback clock, per-request TTFT/TPOT, linear p95, trace context | Azure ML endpoint plus APIM/orchestrator telemetry and Azure Monitor | Export delivery, metric definitions, workload mix, warm-up, autoscale, quota, network, p99, cost |

**Common Pitfalls**

| | Pattern | Why it matters |
|---|---|---|
| Wrong | Gate on mean latency | A small slow tail disappears |
| Right | Gate named populations and percentiles | The release decision matches user-visible failure |
| Wrong | Put trace IDs in metric labels | Cardinality and privacy become operational failures |
| Right | Keep trace IDs in trace context; aggregate bounded metric dimensions | Correlation and metrics serve different jobs |

**Quick Health Check:** require a named request count, success/rejection denominators, mean and p95 side by side, separate TTFT/TPOT, bounded attributes, and an honest branch when any gate fails.

In [ ]:
# ── Measure a Healthy Mean and a Failing Tail ─────────────────────────────
tail_ttft_ms = [18] * 8 + [220] * 2
tail_metrics = []
tail_responses = []
for index, ttft_ms in enumerate(tail_ttft_ms):
    payload = deepcopy(safe_request)
    payload["messages"][-1]["content"] += f" Tail sample {index}."
    service = RiversideService(
        release_manifest,
        SyntheticBackend(f"tail-{index}", first_token_ms=ttft_ms, tpot_ms=2, completion_tokens=6),
        replace(
            local_policy, request_deadline_ms=500, cache_enabled=False,
            singleflight_enabled=False, max_concurrency=1,
        ),
    )
    service.validate_and_warm(release_schema, CONTRACT_DIR)
    server = start_local_server(service)
    adapter = build_endpoint_adapter(local_config, endpoint_override=server.base_url)
    tail_responses.append(adapter.invoke(payload, idempotency_key=f"tail-{index}"))
    server.close()
    tail_metrics.extend(service.metrics)

tail_summary = summarize_metrics(tail_metrics)
tail_mean_ms = mean(metric.total_ms for metric in tail_metrics)
success_rate = sum(response.status == 200 for response in tail_responses) / len(tail_responses)
rejection_rate = sum(response.status == 429 for response in tail_responses) / len(tail_responses)
thresholds = slo_policy["thresholds"]
slo_gates = {
    "minimum_success_rate": success_rate >= thresholds["minimum_success_rate"],
    "maximum_p95_total_ms": tail_summary["p95_total_ms"] <= thresholds["maximum_p95_total_ms"],
    "maximum_p95_ttft_ms": tail_summary["p95_ttft_ms"] <= thresholds["maximum_p95_ttft_ms"],
    "maximum_p95_tpot_ms": tail_summary["p95_tpot_ms"] <= thresholds["maximum_p95_tpot_ms"],
    "maximum_rejection_rate": rejection_rate <= thresholds["maximum_rejection_rate"],
}
print(f"Mean total: {tail_mean_ms:.2f} ms; p95 total: {tail_summary['p95_total_ms']:.2f} ms")
print(f"p95 TTFT: {tail_summary['p95_ttft_ms']:.2f} ms; p95 TPOT: {tail_summary['p95_tpot_ms']:.2f} ms")
for name, passed in slo_gates.items():
    print(f"{'PASS' if passed else 'FAIL'}: {name}")
if tail_mean_ms <= thresholds["maximum_p95_total_ms"] and not slo_gates["maximum_p95_total_ms"]:
    print("Prediction 2 confirmed: the mean looks healthy while p95 blocks the local release.")
else:
    print("Prediction differed: report the measured mean and p95 honestly; do not force the textbook outcome.")
local_slo_decision = "PASS_LOCAL_SLO" if all(slo_gates.values()) else "HOLD_LOCAL_RELEASE"
print(f"Local SLO decision: {local_slo_decision}")
print("LIMIT: ten substituted local requests do not prove p99, capacity, cost, or Azure performance.")

**Checkpoint:** The release decision now uses success, rejection, total latency, TTFT, and TPOT gates instead of one average.

**Reflection:** A local SLO decision is finally possible for the named sample. It still says nothing about Azure ML replicas, APIM policy order, managed identity, network paths, autoscaling, regional capacity, exporter delivery, or cloud cost. The next section makes that boundary executable rather than rhetorical.

## 7 · One Config, Two Endpoint Adapters

The local lab and Azure target share a logical boundary, not an implementation. The same request body and service policy select either loopback HTTP or an Azure ML/APIM request plan. The Azure adapter refuses every network call so source inspection cannot be mistaken for cloud validation.

```mermaid
flowchart LR
    A["OpenAI-compatible request"] --> B{"endpoint_adapter.kind"}
    B -->|local_sandbox| C["Loopback HTTP adapter"]
    C --> D["Measured local mechanisms"]
    B -->|azureml_apim| E["Azure request planner"]
    E --> F["Network call blocked"]
    F --> G["Authorized cloud validation required"]
    style A fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style B fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style C fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style D fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style E fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style F fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style G fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```

**Predict:** If both profiles carry the same service policy and request keys, does that prove A) Azure compatibility, B) contract/config parity only, or C) a live APIM-to-Azure-ML path?

| Mechanism | Local evidence when run | Azure target | Parity intentionally kept | Still unproven |
|---|---|---|---|---|
| Request/response/error contract | Loopback JSON validation | APIM application boundary | Schema and normalized error shape | APIM policy execution and backend translation |
| Artifact/readiness | Fixture invariant plus warm-up | Azure ML deployment probes | Immutable release identity and readiness intent | Registry access, image/model load, probe timing |
| Admission | Semaphore and 429 | APIM/orchestrator plus Azure ML capacity | Retryable overload contract | Replica/token limits, quota, autoscale |
| Idempotency/cache | In-memory single-flight | Shared application/Redis layer | Release-aware key and usage invariant | Cross-replica consistency, TTL, encryption |
| Deadline/resilience | Local clock, retry, breaker, fallback | APIM/orchestrator/backend policy | Absolute-deadline and explicit-fallback intent | Policy ordering, identity, fallback quality |
| Release routing | Seeded blue/green weights | Azure ML deployments plus APIM routing | Stable alias and zero-green rollback intent | Real traffic split, drain, stickiness, rollback time |
| Telemetry/SLO | Local metric records | Application Insights/Azure Monitor | Bounded attribute names and metric definitions | Export, retention, alerts, cloud latency and cost |

**Common Pitfalls**

| | Pattern | Why it matters |
|---|---|---|
| Wrong | Call the local lab an Azure emulator | Control-plane and managed-service behavior is invented |
| Right | Name the substitute and the exact parity contract | Local evidence stays useful without becoming a cloud claim |
| Wrong | Put an API key in the production-shaped fixture | The tutorial teaches secret distribution by accident |
| Right | Plan workload identity and block network traffic | Authentication remains an external validation gate |

**Quick Health Check:** require identical service policy, identical request-body keys, workload-identity intent, reserved `.invalid` endpoint, explicit network block, and a list of cloud-only tests.

In [ ]:
# ── Prove Config Parity While Blocking Azure Network Traffic ──────────────
azure_config = load_json(FIXTURE_DIR / "deployment.azureml-apim.json")
azure_adapter = build_endpoint_adapter(azure_config)
azure_plan = azure_adapter.request_plan(safe_request)
cloud_call_blocked = False
try:
    azure_adapter.invoke(safe_request, idempotency_key="must-not-leave-host")
except CloudCallBlocked:
    cloud_call_blocked = True

adapter_checks = {
    "same_service_policy": local_config["service_policy"] == azure_config["service_policy"],
    "same_request_body_keys": azure_plan["body_keys"] == sorted(safe_request.keys()),
    "workload_identity_planned": azure_plan["authentication"] == "workload_identity",
    "reserved_endpoint": ".invalid/" in azure_plan["url"],
    "plan_marks_network_blocked": azure_plan["network_call"] == "BLOCKED_IN_TUTORIAL",
    "invoke_blocks_network": cloud_call_blocked,
}
print(json.dumps(azure_plan, indent=2))
for name, passed in adapter_checks.items():
    print(f"{'PASS' if passed else 'FAIL'}: {name}")
if all(adapter_checks.values()):
    print("Prediction B confirmed: the profiles prove contract/config parity only; Azure remains unvalidated.")
else:
    print("Prediction contradicted: even the intended local-to-Azure configuration parity is incomplete.")

**Checkpoint:** The same application request and service policy now select two adapters without allowing the tutorial to contact Azure.

**Reflection:** Configuration parity is the final local bridge, not the final release gate. Riverside can now state what the notebook will measure locally and what an authorized cloud run must still prove. The closing decision must keep those evidence classes separate.

In [ ]:
# ── Assemble the Closing Evidence Scorecard ───────────────────────────────
# Every value below references evidence already produced by an earlier cell.
scorecard = {
    "contract_and_readiness": all(lifecycle_checks.values()),
    "bounded_admission": all(admission_checks.values()),
    "dedup_and_accounting": all(dedup_checks.values()),
    "deadline_and_resilience": all(resilience_checks.values()),
    "immutable_release_routing": all(release_checks.values()),
    "local_slo_gates": all(slo_gates.values()),
    "adapter_contract_parity": all(adapter_checks.values()),
}
print("Evidence scope: LOCAL mechanisms | SUBSTITUTED model | UNVALIDATED Azure")
print("=" * 68)
for mechanism, passed in scorecard.items():
    print(f"{mechanism:<34} | {'PASS' if passed else 'HOLD'}")
print("=" * 68)
local_release_decision = "PASS_LOCAL_MECHANISMS" if all(scorecard.values()) else "HOLD_LOCAL_RELEASE"
azure_release_decision = "HOLD_AZURE_PROMOTION"
print(f"Local decision: {local_release_decision}")
print(f"Azure decision: {azure_release_decision}")
if local_release_decision == "PASS_LOCAL_MECHANISMS":
    print("Takeaway: retain the named local evidence, then begin authorized cloud validation.")
else:
    failed_mechanisms = [name for name, passed in scorecard.items() if not passed]
    print(f"Takeaway: repair local evidence first: {failed_mechanisms}")
print("A local pass never overrides missing Azure identity, policy, load, network, monitoring, rollback, quota, region, or cost evidence.")

## 8 · Coverage and Decision

The chapter opened at 0 of 7 authored serving proofs. The scorecard below assembles the same Riverside release's local mechanisms without recomputing them. Because the notebook is committed unexecuted, the table after it describes expected evidence fields, not stored results.

```mermaid
flowchart LR
    A["Seven authored local proofs"] --> B{"All local gates pass?"}
    B -->|no| C["HOLD local release"]
    B -->|yes| D["Local mechanism evidence"]
    D --> E{"Authorized Azure evidence?"}
    E -->|no| F["HOLD Azure promotion"]
    E -->|yes| G["Consider bounded rollout"]
    style A fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style B fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style C fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style D fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style E fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style F fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style G fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```

### Completed roadmap

| Step | Before | Evidence produced when run | Honest decision boundary |
|---|---|---|---|
| Contract/readiness | Function return only | Release/request validation; cold 503; warm 200; mutable version rejected | Local lifecycle only |
| Admission | Ready process accepts every request | Long queue versus 10 ms queue; normalized 429 and retry guidance | Named local workload only |
| Dedup/accounting | Six repeats can create six calls and zero usage | One backend call, five hits, usage invariant, bounded telemetry | In-memory single process |
| Deadline/resilience | Each attempt can spend a fresh timeout | Normalized 504, bounded retry, open circuit, explicit fallback | Synthetic local backend |
| Release control | Mutable active files and no rollback target | Two immutable manifests, seeded canary, zero-green rollback | Routing mechanics, not cloud rollout |
| Tail evidence | Average latency only | Mean, p95 total, TTFT, TPOT, success and rejection gates | Ten substituted local requests |
| Azure bridge | Architecture prose only | Same policy/request keys and a blocked workload-identity request plan | Config parity, not Azure validation |

### When to use what

| Operational signal | First mechanism to inspect |
|---|---|
| Cold or mismatched artifact | Release invariant, readiness, warm-up |
| Queue p95 rises before backend time | Admission budget and concurrency |
| Identical bursts multiply backend calls | Idempotency key and single-flight |
| Recovery exceeds caller budget | Absolute deadline, retry count, breaker |
| Candidate needs reversible exposure | Immutable blue/green routing |
| Mean is healthy but complaints persist | p95 total, TTFT, TPOT, and trace slices |
| Local checks pass but cloud claim is requested | Stop and collect named Azure evidence |

### Three-tier coverage ledger

| Tier | Techniques | Evidence or reason |
|---|---|---|
| Built with proof cells authored | Request/response/error validation; release invariants; readiness/warm-up; bounded concurrency and queue; normalized overload; idempotency; single-flight; approximate token accounting; absolute deadlines; bounded retries; circuit breaker; explicit fallback; immutable blue/green manifests; weighted routing; zero-green rollback; total latency; TTFT; TPOT; p50/p95; bounded telemetry labels; endpoint-adapter factory; blocked Azure request plan | Source is complete but intentionally unexecuted |
| Explained and illustrated | Redis/shared cache; TTL and eviction; SSE streaming; OTel export; trace visualization; graceful drain; shadow and canary observation | Needs external infrastructure, longer workloads, or retained operations evidence |
| Named with a reason | Entra ID token validation; managed identity; APIM policy execution; Azure ML autoscaling/quota/regional capacity; private networking; Azure Monitor delivery/alerts; cloud cost; p99; multi-region recovery | Requires authorized target-environment evidence and cannot be simulated honestly here |

If a technique named above is absent from this ledger, that is exactly the coverage bug this section exists to catch.

### Key takeaways

1. A model artifact becomes serviceable only after contract, lifecycle, and readiness checks.
2. Reject overload while the caller can still recover; a long queue is not capacity.
3. Cache validity and duplicate collapse are separate mechanisms.
4. One absolute deadline owns queueing, retries, fallback, and decode.
5. Rollback needs an immutable target and a traffic action, not a mutable label.
6. Mean latency does not defend the tail; gate total latency, TTFT, and TPOT separately.
7. Local contract parity earns a cloud test plan, never an Azure claim.

### Decision

The authored source supports a future **local mechanism evaluation** only. It does not support promotion: the notebook has no stored measurements, the release manifest's evaluation decision remains `hold`, and every Azure behavior is `UNVALIDATED`. When the notebook is later run, the closing scorecard may still hold the local release if any measured gate fails. Azure promotion remains blocked until authorized smoke, identity, policy, load, monitoring, rollback, networking, quota, regional, and cost evidence is retained.